# Week 2 — MLP Baseline (MVP v0.1)

**目标**：用 `Linear(30→64) → ReLU → Dropout → Linear(64→1)` 在 Kaggle 信用卡欺诈数据上跑通完整训练/评估管线，达成 val **AUC-PR > 0.70**。

**产出**：
- 完整训练循环（分层切分、early stopping、best-model checkpoint）
- 不平衡处理：`BCEWithLogitsLoss(pos_weight=...)`
- 评估：AUC-PR、AUC-ROC、Recall@FPR=0.001、混淆矩阵
- Focal Loss 可切换版本（对比加权 BCE）

**核心原则**：能解释每一行代码。

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')


## 1. 下载 Kaggle 数据（复用 Week 1 的模式）

凭证来源：Colab Secrets 的 `KAGGLE_USERNAME` / `KAGGLE_KEY`。本地运行请确保 shell env 导出同名变量或 `~/.kaggle/kaggle.json` 存在。

In [ ]:
import pathlib

DATA_DIR = PROJECT_ROOT / 'data'
CSV = DATA_DIR / 'creditcard.csv'

if not CSV.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !pip install -q kaggle
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p {DATA_DIR} --unzip

print('Data ready at:', CSV)

In [ ]:
import pandas as pd

df = pd.read_csv(CSV)
print('shape:', df.shape)
print('fraud ratio:', df['Class'].mean())
df.head(3)

## 2. 数据切分 + 标准化

**关键点**：
- **分层切分**（stratify）保证每个 split 的欺诈比例一致；欺诈只占 0.17%，随机切会让 val/test 的正样本极不稳定。
- **Scaler 只 fit 在训练集上**：避免把 val/test 的统计量泄露进训练。
- `Time` 列单位是秒，`Amount` 列未标准化 → 明显会让梯度爆炸，所以和 `V1–V28` 一起过 `StandardScaler`。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols].values.astype('float32')
y = df['Class'].values.astype('float32')

# 分层切分：先切出 test，再从剩余里切 val
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765,  # 0.15 / 0.85
    stratify=y_trainval, random_state=SEED)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype('float32')
X_val   = scaler.transform(X_val).astype('float32')
X_test  = scaler.transform(X_test).astype('float32')

print(f'train: {X_train.shape} fraud={y_train.mean():.5f}')
print(f'val  : {X_val.shape}   fraud={y_val.mean():.5f}')
print(f'test : {X_test.shape}  fraud={y_test.mean():.5f}')

## 3. PyTorch Dataset / DataLoader

简单得不用子类化 `Dataset`，`TensorDataset` 就够了；但我们还是写一版显式的 `FraudDataset`，方便 Week 3 扩展到序列。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH = 512
train_ds = FraudDataset(X_train, y_train)
val_ds   = FraudDataset(X_val, y_val)
test_ds  = FraudDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)
print('batches:', len(train_loader), len(val_loader), len(test_loader))

## 4. 模型：MLP

结构：`Linear(F→64) → ReLU → Dropout(0.3) → Linear(64→64) → ReLU → Dropout(0.3) → Linear(64→1)`。
输出 1 维 logit，不接 sigmoid —— `BCEWithLogitsLoss` 内部会做 log-sum-exp，比 `Sigmoid + BCELoss` 数值更稳定。

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=64, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)  # (B,)

IN_DIM = X_train.shape[1]
model = MLP(IN_DIM).to(device)
print(model)
print('params:', sum(p.numel() for p in model.parameters()))

## 5. 损失函数：加权 BCE

欺诈 : 正常 ≈ 1 : 577。直接 BCE 会让模型几乎永远预测"正常"。
用 `pos_weight = neg / pos` 放大正样本的梯度，让 loss 对欺诈敏感。

> 如果想对比 Focal Loss，见第 10 节的可切换 cell。

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
print(f'neg={neg}, pos={pos}, pos_weight={pos_weight.item():.1f}')

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## 6. 评估指标

- **AUC-PR** (`average_precision_score`)：PR 曲线下面积，极度不平衡下比 AUC-ROC 更有区分度。
- **AUC-ROC**：参考对比。
- **Recall@FPR=0.001**：业务语义——"每放行 1000 个正常交易最多误拦 1 个"时的召回率。
- **混淆矩阵**：阈值取让 val F1 最大的点。

In [ ]:
import numpy as np
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_recall_curve, confusion_matrix, roc_curve)

@torch.no_grad()
def predict_scores(model, loader):
    model.eval()
    scores, labels = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        scores.append(torch.sigmoid(logits).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(scores), np.concatenate(labels)

def evaluate(model, loader, name='val'):
    s, y = predict_scores(model, loader)
    ap = average_precision_score(y, s)
    roc = roc_auc_score(y, s)
    # Recall @ FPR = 0.001
    fpr, tpr, thr = roc_curve(y, s)
    idx = np.searchsorted(fpr, 0.001)
    rec_at_fpr = tpr[min(idx, len(tpr) - 1)]
    return {'ap': ap, 'roc': roc, 'recall@fpr=0.001': rec_at_fpr, 'scores': s, 'labels': y}

## 7. 训练循环 + Early Stopping + Best-Model Checkpoint

5 个关键组件：
1. `optimizer.zero_grad()` — 清梯度
2. 前向 + loss
3. `loss.backward()` — 反向
4. `optimizer.step()` — 更新
5. 监控 val 指标，保存最好的那次权重

Early stopping patience=5，监控 `val AUC-PR`。

In [ ]:
import copy, time

def train_model(model, train_loader, val_loader, loss_fn,
                lr=1e-3, weight_decay=1e-5, max_epochs=30, patience=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_ap, best_state, bad = -1.0, None, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item() * x.size(0)
        train_loss = total / len(train_loader.dataset)

        val = evaluate(model, val_loader)
        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_ap': val['ap'], 'val_roc': val['roc']})
        print(f"epoch {epoch:02d}  loss={train_loss:.4f}  val_ap={val['ap']:.4f}  "
              f"val_roc={val['roc']:.4f}  recall@fpr=0.001={val['recall@fpr=0.001']:.3f}  "
              f"({time.time()-t0:.1f}s)")

        if val['ap'] > best_ap:
            best_ap = val['ap']
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f'early stop @ epoch {epoch}, best val_ap={best_ap:.4f}')
                break

    model.load_state_dict(best_state)
    return model, history, best_ap

In [ ]:
torch.manual_seed(SEED)
model = MLP(IN_DIM).to(device)
model, history, best_ap = train_model(model, train_loader, val_loader, loss_fn,
                                      lr=1e-3, max_epochs=30, patience=5)
print(f'\nbest val AUC-PR = {best_ap:.4f}')

## 8. 保存 Best Model + 最终 Test 评估

In [ ]:
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'mlp_baseline.pt'
ckpt_path.parent.mkdir(exist_ok=True, parents=True)
torch.save({'state_dict': model.state_dict(),
            'in_dim': IN_DIM,
            'scaler_mean': scaler.mean_,
            'scaler_scale': scaler.scale_}, ckpt_path)
print('saved to', ckpt_path)

test = evaluate(model, test_loader, name='test')
print(f"test AUC-PR  = {test['ap']:.4f}")
print(f"test AUC-ROC = {test['roc']:.4f}")
print(f"test Recall@FPR=0.001 = {test['recall@fpr=0.001']:.3f}")

## 9. 混淆矩阵（阈值取最大 F1）

In [ ]:
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt, seaborn as sns

s, y_true = test['scores'], test['labels']
prec, rec, thr = precision_recall_curve(y_true, s)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.nanargmax(f1[:-1])
best_thr = thr[best_idx]
y_pred = (s >= best_thr).astype(int)

cm = confusion_matrix(y_true, y_pred)
print(f'best threshold = {best_thr:.4f}  F1 = {f1[best_idx]:.3f}')
print('confusion matrix:\n', cm)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['pred 0', 'pred 1'], yticklabels=['true 0', 'true 1'])
ax[0].set_title('Confusion matrix (threshold = max-F1)')

ax[1].plot(rec, prec)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision')
ax[1].set_title(f'PR curve (AUC-PR = {test["ap"]:.3f})')
plt.tight_layout()

## 10. 可选：Focal Loss 对比

**动机**：加权 BCE 对所有正样本一视同仁地放大梯度；Focal Loss 额外降低"易分样本"的权重 `(1-p_t)^γ`，让模型更聚焦难分样本。

公式：$FL(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$，其中 $p_t = p$ (y=1) 或 $1-p$ (y=0)。

把 `USE_FOCAL = True` 并重跑本 cell 看效果。

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * ce).mean()

USE_FOCAL = False   # ← 设为 True 来跑对比
if USE_FOCAL:
    torch.manual_seed(SEED)
    model_fl = MLP(IN_DIM).to(device)
    loss_fl = FocalLoss(alpha=0.25, gamma=2.0)
    model_fl, hist_fl, ap_fl = train_model(model_fl, train_loader, val_loader, loss_fl,
                                           lr=1e-3, max_epochs=30, patience=5)
    test_fl = evaluate(model_fl, test_loader)
    print(f"[focal]   test AUC-PR = {test_fl['ap']:.4f}")
    print(f"[weighted] test AUC-PR = {test['ap']:.4f}")
else:
    print('Focal Loss 未启用；设 USE_FOCAL=True 重跑本 cell 对比。')

## 11. 本周复盘笔记

1. **训练循环 5 个关键组件**：`zero_grad → forward → loss → backward → step`（外加 clip + eval + early stop）。
2. **为什么用 `pos_weight` 而不是过采样 SMOTE？** `pos_weight` 改变的是梯度幅度，不制造虚假样本，baseline 阶段更干净。
3. **AUC-PR vs AUC-ROC 在本数据上的差距**：_（填：ROC 通常偏高 0.02~0.05，PR 更反映业务难度。）_
4. **加权 BCE vs Focal Loss 观察**：_（填：Focal 在难样本多的场景更强；信用卡 PCA 特征已经很干净，差距可能不大。）_
5. **下周要做什么？** 把单笔映射到"按用户+时间排序的序列"，准备 LSTM baseline。